<a href="https://colab.research.google.com/github/somayahalbaradei/nora/blob/main/notebooks/NORA_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# NORA — evidence validation demo

Runs on GSE271389: 47 plasma cfRNA nanopore libraries.

**Cell 1** needs no API key. It reproduces the core finding with statistics alone.

**Cell 2** runs the agent, and needs an Anthropic API key.


In [ ]:
!git clone -q https://github.com/somayahalbaradei/nora.git 2>/dev/null || true
%cd nora
!pip install -q -r requirements.txt

## 1. Design check — no API key needed

Would this cohort have been worth sequencing?

In [ ]:
!python -m nora.design --check data/gse271389_manifest.tsv --group group --position well

## 2. The finding, by hand

Fragment length looks like a biomarker. Then you condition on sequencing run.

In [ ]:
import importlib.util, pandas as pd
spec = importlib.util.spec_from_file_location('na','nora/agent.py')
na = importlib.util.module_from_spec(spec); spec.loader.exec_module(na)
na.DATA = pd.read_csv('data/nora_data.tsv', sep='\t')

print(na.derive_variable('plate_row','title','first_char'))
print()
print('N50 by diagnosis :', na.test_association('n50','group'))
print('N50 by run       :', na.test_association('n50','plate_row'))
print()
print(na.condition_and_retest('n50','group','plate_row'))

## 3. Tools that refuse

Diagnosis is completely confounded with plate column. `scipy` would return a
p-value here. NORA declines.

In [ ]:
na.derive_variable('plate_col','title','numeric_part')
print(na.crosstab('plate_col','group'))
print()
print(na.stratified_permutation('mean_len','group','plate_col'))

## 4. What does chance look like here?

Selection outside the cross-validation loop, on data containing no signal.

In [ ]:
print('leaky  :', na.simulate_null([16,12,19], leaky=True,  nsim=20))
print('nested :', na.simulate_null([16,12,19], leaky=False, nsim=20))

## 5. The agent — needs an API key

It is not told that A01–D12 encodes plate position.

In [ ]:
import os
os.environ['ANTHROPIC_API_KEY'] = ''  # paste key here
!python -m nora.agent --ask "Is fragment length a valid biomarker in this cohort?" --data data/nora_data.tsv